In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import networkx as nx
import glob
import json
import os
import shutil

from sidion_qubo import load_data, get_G_ising, Ising

In [23]:
solver_name = "Advantage2_system1.8"

with open(f"../data/topologies/{solver_name}_topology.json", "r") as f:
    topology = json.load(f)

active_qubits = topology["active_qubits"] 
active_couplers = [tuple(sorted(e)) for e in topology["active_couplers"]]

In [24]:
case_path = '../data/original_miha_instances/send_{}/m{}/case{}/beta{}/'
list_of_cases = sorted(glob.glob(case_path.format('*', '*', '*', 0.0)))
isings = ["J_log", "J_qac"]


def remove_missing_qubits_and_couplers(ising, active_qubits, active_couplers):
    ising_copy = ising.copy()
    not_available_qubits = set(ising_copy.nodes)-set(active_qubits)
    if not_available_qubits:
        print('Removing qubits:', not_available_qubits)
        ising_copy.remove_nodes_from(not_available_qubits)

    ising_copy_edges = (tuple(sorted(e)) for e in ising_copy.edges)
    not_available_couplers = set(ising_copy_edges)-set(active_couplers)
    if not_available_couplers:
        print('Removing couplers:', not_available_couplers)
        ising_copy.remove_edges_from(not_available_couplers)

    return ising_copy

for case in list_of_cases:
    data = load_data(case, isings)
    ising_qac = get_G_ising(
        data["J_qac"],
        info={'type': 'J_qac', 'case': case, 'topology': solver_name}
    )

    ising_qac = remove_missing_qubits_and_couplers(ising_qac, active_qubits, active_couplers)
    ising_qac.remove_nodes_from(data["pen"])

    
    ising_log = Ising(
        nx.induced_subgraph(ising_qac, data["n3"]),
        info={'type': 'J_log (n3)', 'case': 'case', 'topology': solver_name}
    )
    

    new_case_path = case.replace('original_miha_instances', f'{solver_name}')
    os.makedirs(os.path.dirname(new_case_path), exist_ok=True)

    with open(f"{new_case_path}/J_log.json", "w") as f:
        json.dump(ising_log.serialize(), f, indent=4)

    with open(f"{new_case_path}/J_qac.json", "w") as f:
        json.dump(ising_qac.serialize(), f, indent=4)

    shutil.copyfile(f"{case}/graph_all.txt", f"{new_case_path}/graph_all.txt")
    
    



Loading data from: ../data/original_miha_instances/send_8/m1/case0/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case1/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case2/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case3/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case4/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case5/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case6/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case7/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case8/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m1/case9/beta0.0/ ...
... Done.
Loading data from: ../data/original_miha_instances/send_8/m10/case0/beta0.0/ ...
... Done.
Removing 